In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from brain_image.model.comm_alignment import CommAlignmentModel
from brain_image.model.prior import SimpleDiffusionPrior

from pathlib import Path

/home/gasparyanartur/dev/brain-image-implementation/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/gasparyanartur/dev/brain-image-implementation/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadat

In [3]:
comm_cp = Path("checkpoints/train_comm-15445420-20260206_021738/version_0/checkpoints/epoch_0033-val_loss_17.7101.ckpt")
prior_cp = Path("checkpoints/train_eeg-15445445-20260206_025856/version_0/checkpoints/epoch_0484-val_loss_0.4309.ckpt")

In [4]:
comm = CommAlignmentModel.load_from_checkpoint(comm_cp)

Seed set to 42


In [5]:
comm.data_module.get_embedding_stats()["align_img_latent"]

{'mean': tensor([ 2.0597e-02,  3.5471e-01, -1.9733e-01, -5.4356e-01,  2.5470e-02,
         -4.3438e-01,  1.9933e-01, -2.1577e-02,  8.2954e-02,  1.3881e-01,
          6.3127e-01, -6.1978e-01,  1.2303e-01, -3.3305e-01,  2.2397e-01,
         -1.5610e-01,  2.2058e-01, -1.9588e-01, -2.2757e-01, -6.6696e-01,
          1.2817e-02, -3.0341e-01, -2.7007e-01, -1.4193e-01, -2.2645e-01,
          2.2633e-01, -3.1258e-01,  3.9390e-01,  4.0965e-02, -1.4023e-01,
          8.2399e-02, -2.4180e-01, -2.0883e-01,  1.4181e-01, -3.7954e-01,
         -3.3417e-02, -8.6429e-02,  4.2923e-01,  1.3293e-01, -1.6592e-01,
         -2.2331e-01,  6.4705e-01, -2.8378e-01, -5.9106e-02, -1.7561e-01,
         -3.2702e-02, -1.3271e-01, -3.5343e-02,  5.7232e-01, -1.8031e-01,
         -9.3111e-03, -1.0926e-01, -1.3689e-01, -1.8009e-01, -2.2037e-02,
          4.4725e-01, -1.2442e-01,  9.6313e-02,  3.3259e-01, -1.2600e-02,
         -1.4922e-01, -2.7564e-02,  7.9709e-02,  2.0239e-01, -2.0249e-01,
         -2.0215e-01,  2.3934e

In [6]:
import torch

state_dict = torch.load(prior_cp)
state_dict

{'epoch': 484,
 'global_step': 8245,
 'pytorch-lightning_version': '2.5.5',
 'state_dict': OrderedDict([('prior.input_proj.linear.weight',
               tensor([[ 0.0128,  0.0633, -0.0375,  ...,  0.0250,  0.0065, -0.0415],
                       [-0.1183,  0.0968, -0.0238,  ...,  0.0182,  0.0251, -0.0141],
                       [ 0.0470, -0.0039, -0.0393,  ...,  0.0023, -0.0199, -0.0081],
                       ...,
                       [-0.0259,  0.0410,  0.0388,  ...,  0.1089, -0.0642, -0.0530],
                       [ 0.0053,  0.0195,  0.0278,  ..., -0.0111,  0.0091, -0.0394],
                       [-0.0318,  0.0389,  0.0285,  ...,  0.0123,  0.0559, -0.0397]],
                      device='cuda:0')),
              ('prior.input_proj.linear.bias',
               tensor([ 0.2538, -0.1175, -0.0667,  ..., -0.1701, -0.1409, -0.2302],
                      device='cuda:0')),
              ('prior.input_proj.norm.weight',
               tensor([0.6613, 0.4200, 0.4448,  ..., 0.4141, 0

In [7]:
prior_config = state_dict["hyper_parameters"]["config"]["prior"]
prior_config

{'d_input': 768,
 'd_cond': 768,
 'd_time': 512,
 'd_hidden_start': 1024,
 'd_hidden_scale': 0.5,
 'depth': 5,
 'act_func': 'silu',
 'dropout': 0.1,
 'cond_drop_prob': 0.1,
 'norm_scheme': 'z_scale',
 'layer_norm_out': False,
 'num_training_timesteps': 1000}

In [8]:


#state_dict["state_dict"].keys()
from brain_image.utils import get_submodules_with_pattern

prior_state_dict = get_submodules_with_pattern(state_dict["state_dict"], "prior")
prior_state_dict

{'input_proj.linear.weight': tensor([[ 0.0128,  0.0633, -0.0375,  ...,  0.0250,  0.0065, -0.0415],
         [-0.1183,  0.0968, -0.0238,  ...,  0.0182,  0.0251, -0.0141],
         [ 0.0470, -0.0039, -0.0393,  ...,  0.0023, -0.0199, -0.0081],
         ...,
         [-0.0259,  0.0410,  0.0388,  ...,  0.1089, -0.0642, -0.0530],
         [ 0.0053,  0.0195,  0.0278,  ..., -0.0111,  0.0091, -0.0394],
         [-0.0318,  0.0389,  0.0285,  ...,  0.0123,  0.0559, -0.0397]],
        device='cuda:0'),
 'input_proj.linear.bias': tensor([ 0.2538, -0.1175, -0.0667,  ..., -0.1701, -0.1409, -0.2302],
        device='cuda:0'),
 'input_proj.norm.weight': tensor([0.6613, 0.4200, 0.4448,  ..., 0.4141, 0.4235, 0.3934], device='cuda:0'),
 'input_proj.norm.bias': tensor([1.2557, 1.3056, 1.2748,  ..., 1.3123, 1.2803, 1.2741], device='cuda:0'),
 'time_encoders.0.linear_1.weight': tensor([[-0.0096, -0.0021, -0.0032,  ..., -0.0401, -0.0018, -0.0153],
         [ 0.0086, -0.0011, -0.0177,  ...,  0.0242,  0.0197, -0

In [9]:
state_dict["state_dict"].keys()

odict_keys(['prior.input_proj.linear.weight', 'prior.input_proj.linear.bias', 'prior.input_proj.norm.weight', 'prior.input_proj.norm.bias', 'prior.time_encoders.0.linear_1.weight', 'prior.time_encoders.0.linear_1.bias', 'prior.time_encoders.0.linear_2.weight', 'prior.time_encoders.0.linear_2.bias', 'prior.time_encoders.1.linear_1.weight', 'prior.time_encoders.1.linear_1.bias', 'prior.time_encoders.1.linear_2.weight', 'prior.time_encoders.1.linear_2.bias', 'prior.time_encoders.2.linear_1.weight', 'prior.time_encoders.2.linear_1.bias', 'prior.time_encoders.2.linear_2.weight', 'prior.time_encoders.2.linear_2.bias', 'prior.time_encoders.3.linear_1.weight', 'prior.time_encoders.3.linear_1.bias', 'prior.time_encoders.3.linear_2.weight', 'prior.time_encoders.3.linear_2.bias', 'prior.time_encoders.4.linear_1.weight', 'prior.time_encoders.4.linear_1.bias', 'prior.time_encoders.4.linear_2.weight', 'prior.time_encoders.4.linear_2.bias', 'prior.cond_encoders.0.weight', 'prior.cond_encoders.0.bias'

In [12]:
from brain_image.model.prior import DiffusionPriorConfig


prior = SimpleDiffusionPrior(
    DiffusionPriorConfig(**prior_config),
    embedding_stats=comm.data_module.get_embedding_stats()["align_img_latent"]
)
prior.load_state_dict(prior_state_dict)

<All keys matched successfully>

In [13]:
prior.embedding_stats

{'mean': tensor([ 2.0597e-02,  3.5471e-01, -1.9733e-01, -5.4356e-01,  2.5470e-02,
         -4.3438e-01,  1.9933e-01, -2.1577e-02,  8.2954e-02,  1.3881e-01,
          6.3127e-01, -6.1978e-01,  1.2303e-01, -3.3305e-01,  2.2397e-01,
         -1.5610e-01,  2.2058e-01, -1.9588e-01, -2.2757e-01, -6.6696e-01,
          1.2817e-02, -3.0341e-01, -2.7007e-01, -1.4193e-01, -2.2645e-01,
          2.2633e-01, -3.1258e-01,  3.9390e-01,  4.0965e-02, -1.4023e-01,
          8.2399e-02, -2.4180e-01, -2.0883e-01,  1.4181e-01, -3.7954e-01,
         -3.3417e-02, -8.6429e-02,  4.2923e-01,  1.3293e-01, -1.6592e-01,
         -2.2331e-01,  6.4705e-01, -2.8378e-01, -5.9106e-02, -1.7561e-01,
         -3.2702e-02, -1.3271e-01, -3.5343e-02,  5.7232e-01, -1.8031e-01,
         -9.3111e-03, -1.0926e-01, -1.3689e-01, -1.8009e-01, -2.2037e-02,
          4.4725e-01, -1.2442e-01,  9.6313e-02,  3.3259e-01, -1.2600e-02,
         -1.4922e-01, -2.7564e-02,  7.9709e-02,  2.0239e-01, -2.0249e-01,
         -2.0215e-01,  2.3934e

In [14]:
example_batch = next(iter(comm.data_module.get_dataloader("test")))
example_batch.keys()

dict_keys(['img_path', 'eeg_data', 'idx', 'sub', 'align_img_latent'])

In [15]:
import numpy as np
from brain_image.data.data import get_from_batch
from torch import Tensor
from torch.nn import functional as F
import itertools as it
import tqdm
import pandas as pd


device = torch.device("cuda")
prior.eval()
prior.to(device)
comm.eval()
comm.to(device)



CommAlignmentModel(
  (img_encoder): Identity()
  (image_augmenter): LatentAugmentationPipeline(
    (augment_modules): ModuleList(
      (0): WrapAugment(
        (augmentation): LatentAffine()
      )
      (1): WrapAugment(
        (augmentation): LatentNoise()
      )
      (2): WrapAugment(
        (augmentation): LatentDropout()
      )
    )
  )
  (eeg_encoder): AtmsEEGEncoder(
    (encoder): iTransformer(
      (enc_embedding): DataEmbedding(
        (value_embedding): Linear(in_features=250, out_features=250, bias=True)
        (position_embedding): PositionalEmbedding()
        (temporal_embedding): TimeFeatureEmbedding(
          (embed): Linear(in_features=4, out_features=250, bias=False)
        )
        (dropout): Dropout(p=0.25, inplace=False)
      )
      (encoder): Encoder(
        (attn_layers): ModuleList(
          (0): EncoderLayer(
            (attention): AttentionLayer(
              (inner_attention): FullAttention(
                (dropout): Dropout(p=0.25, 

In [ ]:

values = []

ns = list(np.linspace(0, 30, 20).astype(int))
gs = list(np.linspace(0, 4, 20))
params = list(it.product(ns, gs))

for n, g in tqdm.tqdm(params):
    with torch.no_grad():
        target = get_from_batch("align_img_latent", example_batch, Tensor).to(device)
        eeg = get_from_batch("eeg_data", example_batch, Tensor).to(device)
        eeg_latent = F.normalize(comm.eeg_encoder(eeg), dim=-1)
        clip_pred = prior.generate(conditioning=eeg_latent, guidance_scale=g, num_steps=n)
        mean_sim = round(F.cosine_similarity(target, clip_pred, dim=-1).mean().item(), 5)
        std_sim = round(F.cosine_similarity(target, clip_pred, dim=-1).std().item(), 5)
        
        values.append((n, g, mean_sim, std_sim))


df = pd.DataFrame(values, columns=["n", "g", "mean_sim", "std_sim"])
df = df.sort_values("mean_sim", ascending=False)
df

  0%|          | 0/400 [00:00<?, ?it/s]/home/gasparyanartur/dev/brain-image-implementation/.venv/lib/python3.12/site-packages/diffusers/schedulers/scheduling_ddpm.py:307: RuntimeWarning: divide by zero encountered in scalar floor_divide
  step_ratio = self.config.num_train_timesteps // self.num_inference_steps
100%|██████████| 400/400 [00:33<00:00, 11.78it/s]


,n,g,mean_sim,std_sim
65,4,1.052632,0.76876,0.07987
106,7,1.263158,0.76847,0.07748
110,7,2.105263,0.76841,0.08012
67,4,1.473684,0.76770,0.08357
86,6,1.263158,0.76735,0.07939
...,...,...,...,...
8,0,1.684211,0.49638,0.05855
6,0,1.263158,0.49628,0.05831
13,0,2.736842,0.49540,0.05902
4,0,0.842105,0.49484,0.06149


In [106]:
#torch.manual_seed(42)
from brain_image.metrics import get_retrieval_accuracy

with torch.no_grad():
    target = get_from_batch("align_img_latent", example_batch, Tensor).to(device)
    eeg = get_from_batch("eeg_data", example_batch, Tensor).to(device)
    eeg_latent = comm.eeg_encoder(eeg)
    eeg_latent_normed = F.normalize(eeg_latent, dim=-1)
    clip_pred = prior.generate(conditioning=eeg_latent_normed, guidance_scale=1.0, num_steps=15)

    z_proto_gt = comm.comm.encode_feature([eeg_latent, target], [comm.config.eeg_idx, comm.config.img_idx], skip_encoder=True)
    z_proto_pred = comm.comm.encode_feature([eeg_latent, clip_pred], [comm.config.eeg_idx, comm.config.img_idx], skip_encoder=True)
    z_img_target = comm.comm.encode_feature([target], comm.config.img_idx, skip_encoder=True)

    acc_proto_to_img, acc_img_to_proto = get_retrieval_accuracy(z_proto_pred, z_img_target, norm=True)
    acc_proto_to_img_target, acc_img_to_proto_target = get_retrieval_accuracy(z_proto_gt, z_img_target, norm=True)
    print(f"acc_proto_to_img: {acc_proto_to_img}")
    print(f"acc_img_to_proto: {acc_img_to_proto}")

    print(f"acc_proto_to_img_target: {acc_proto_to_img_target}")
    print(f"acc_img_to_proto_target: {acc_img_to_proto_target}")

    print(F.cosine_similarity(z_proto_pred, z_proto_gt, dim=-1).mean().item())

acc_proto_to_img: 0.28999999165534973
acc_img_to_proto: 0.3649999797344208
acc_proto_to_img_target: 0.8849999904632568
acc_img_to_proto_target: 0.98499995470047
0.8226029872894287
